<a href="https://colab.research.google.com/github/ganeshkumarjer-bot/PRODIGY_ML_03/blob/main/SVM_for_C%26Dog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import cv2  # Run: pip install opencv-python

# ==========================================
# 1. PARAMETERS & CONFIGURATION
# ==========================================
# Update these paths to where your actual "train" images folder is saved locally
# The folder should contain images named like "cat.0.jpg", "dog.1.jpg", etc.
IMAGE_DIR = "/content/sampleSubmission.csv"
IMG_SIZE = 64  # Resize to 64x64 to balance performance and feature size
MAX_IMAGES = 2000  # Limiting sample size for quick training execution

# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_preprocess_data(image_dir, img_size, max_images):
    X = []
    y = []

    # List all files in the directory
    all_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    # Shuffle or slice to take a manageable sample
    np.random.seed(42)
    np.random.shuffle(all_files)
    selected_files = all_files[:max_images]

    print(f"Loading and preprocessing {len(selected_files)} images...")

    for filename in selected_files:
        # Determine label based on filename pattern (Kaggle Cats vs Dogs standard format)
        if 'cat' in filename.lower():
            label = 0  # Cat
        elif 'dog' in filename.lower():
            label = 1  # Dog
        else:
            continue  # Skip files that don't match the criteria

        img_path = os.path.join(image_dir, filename)

        try:
            # Load in grayscale to reduce structural feature dimension size
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            # Resize image to a unified resolution
            img_resized = cv2.resize(img, (img_size, img_size))
            # Flatten the 2D array into a 1D vector (64x64 = 4096 features)
            img_flattened = img_resized.flatten()

            X.append(img_flattened)
            y.append(label)
        except Exception as e:
            # Ignore broken/corrupted file exceptions
            continue

    return np.array(X), np.array(y)

# Simulate or invoke data loading (Uncomment locally when images are present)
# X, y = load_and_preprocess_data(IMAGE_DIR, IMG_SIZE, MAX_IMAGES)

# For structural execution tracking in this snippet, let's generate mock dataset matrices
print("Generating mock image arrays to verify pipeline structure...")
X_mock = np.random.randint(0, 256, size=(1000, IMG_SIZE * IMG_SIZE))
y_mock = np.random.randint(0, 2, size=(1000,))

# ==========================================
# 3. SPLITTING AND SCALING
# ==========================================
# Partition into Training (80%) and Test (20%) subsets
X_train, X_test, y_train, y_test = train_test_split(X_mock, y_mock, test_size=0.2, random_state=42)

# Standardize features (Z-score normalization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(float))
X_test_scaled = scaler.transform(X_test.astype(float))

# ==========================================
# 4. TRAINING THE SVM MODEL
# ==========================================
print("Training the Support Vector Machine Classifier...")
# Using an RBF (Radial Basis Function) non-linear kernel
svm_classifier = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_classifier.fit(X_train_scaled, y_train)

# ==========================================
# 5. EVALUATION
# ==========================================
y_pred = svm_classifier.predict(X_test_scaled)

print("\n================= SVM CLASSIFICTION REPORT =================")
print(f"Test Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("============================================================")
print("\nDetailed Metrics:")
print(classification_report(y_test, y_pred, target_names=['Cat (0)', 'Dog (1)']))

Generating mock image arrays to verify pipeline structure...
Training the Support Vector Machine Classifier...

================= SVM CLASSIFICTION REPORT =================
Test Accuracy Score: 52.50%

Detailed Metrics:
              precision    recall  f1-score   support

     Cat (0)       0.47      0.07      0.13        94
     Dog (1)       0.53      0.92      0.67       106

    accuracy                           0.53       200
   macro avg       0.50      0.50      0.40       200
weighted avg       0.50      0.53      0.42       200

